In [1]:
import pandas as pd
import ast
import re

In [2]:
filepath = "../data/raw/vacancies_2026-04-14.csv"
date_str = re.search(r'\d{4}-\d{2}-\d{2}', filepath).group()
df = pd.read_csv(f"../data/raw/vacancies_{date_str}.csv")
print(df.shape)
df.head(3)

(1524, 17)


,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"['ON_SITE', 'HYBRID']",['Дата-сайентист'],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer
1,131823118,9462.6,NaN,Астана,between1And3,full,remote,['REMOTE'],['Специалист технической поддержки'],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer
2,130648164,400000.0,800000.0,Астана,between1And3,full,fullDay,['ON_SITE'],['Дата-сайентист'],False,False,False,False,False,2026-03-20 15:49:17+03:00,Обязанности: • Разработка и улучшение NLP/LLM ...,ML Engineer (NLP / RAG)


In [3]:
df[(df['salary_from']<100_000) | (df['salary_to']<100_000)]

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
1,131823118,9462.60,NaN,Астана,between1And3,full,remote,['REMOTE'],['Специалист технической поддержки'],False,False,True,False,True,2026-04-03 15:03:43+03:00,"Привет! Мы, ""Aspirity Solution"", студия разраб...",Customer Support Engineer
64,131901002,9195.00,18390.00,Астана,between3And6,part,remote,['REMOTE'],"['BI-аналитик, аналитик данных']",False,False,True,False,False,2026-04-13 10:07:31+03:00,Обязанности: — Тестирование продуктовых фич с ...,Power BI Analyst (Product Testing)
763,131927036,4731.30,9462.60,Астана,moreThan6,project,remote,['REMOTE'],"['Программист, разработчик']",False,False,True,False,False,2026-04-07 17:17:05+03:00,Мы ищем опытного Full-Stack разработчика для у...,"AI-first Full Stack Developer (Node.js, React,..."
951,131632083,NaN,11081.60,Алматы,between3And6,project,fullDay,['ON_SITE'],"['Программист, разработчик']",False,False,True,True,False,2026-04-12 16:00:53+03:00,🔧 Программист промышленных роботов (ABB / Fanu...,Инженер-программист (АВВ/FANUC) с ВНЖ Финляндии
1435,131321300,2365.65,2365.65,Астана,between1And3,full,remote,['REMOTE'],['Тестировщик'],False,False,False,False,False,2026-04-11 06:05:53+03:00,"HDCart LIMITED – e-commerce компания, которая ...",QA Engineer (Tbilisi)


In [4]:
# removed non-it vacancies
df.drop(df[df['id'] == 131823118].index, inplace=True)

# we have some hours-paid vacancies, so we should convert them to monthly salary
df.loc[df['id'] == 131898390, ['salary_from', 'salary_to']] *= 4*5*4.33
df.loc[df['id'] == 131927036, ['salary_from', 'salary_to']] *= 6*5*4.33
df.loc[df['id'] == 131632083, ['salary_from', 'salary_to']] *= 8*5*4.33
df.loc[df['id'] == 131321300, ['salary_from', 'salary_to']] *= 8*5*4.33

In [5]:
df[(df['salary_from']<100_000) | (df['salary_to']<100_000)]

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name
64,131901002,9195.0,18390.0,Астана,between3And6,part,remote,['REMOTE'],"['BI-аналитик, аналитик данных']",False,False,True,False,False,2026-04-13 10:07:31+03:00,Обязанности: — Тестирование продуктовых фич с ...,Power BI Analyst (Product Testing)


In [6]:
df['salary_mid'] = df[['salary_from', 'salary_to']].mean(axis=1)

In [7]:
print('Salary stats:')
print(df['salary_mid'].describe().apply(lambda x: f"{x:_.0f}"))

Salary stats:
count          486
mean       654_045
std        490_506
min         13_792
25%        350_000
50%        500_000
75%        750_000
max      3_311_910
Name: salary_mid, dtype: str


In [8]:
df[['name', 'area', 'salary_from', 'salary_to', 'salary_mid', 'professional_role']].sort_values('salary_mid').head(30)

,name,area,salary_from,salary_to,salary_mid,professional_role
64,Power BI Analyst (Product Testing),Астана,9195.0,18390.00,13792.50,"['BI-аналитик, аналитик данных']"
537,Junior разработчик,Алматы,100000.0,NaN,100000.00,"['Программист, разработчик']"
604,Стажер-программист 1С,Астана,100000.0,NaN,100000.00,"['Программист, разработчик']"
1052,Game Development International Internship,Алматы,118282.5,NaN,118282.50,['Гейм-дизайнер']
816,Game Development International Intern,Караганда,118282.5,NaN,118282.50,['Гейм-дизайнер']
1373,Системный администратор,Алматы,110000.0,150000.00,130000.00,['Системный администратор']
1378,Администратор-СММ в детском центре,Астана,100000.0,180000.00,140000.00,['Администратор']
1492,Project Management Trainee,Алматы,NaN,142885.26,142885.26,['Менеджер по персоналу']
1308,Помощник системного администратора,Астана,150000.0,NaN,150000.00,['Системный администратор']
1406,Администратор Bitrix24,Тараз,150000.0,NaN,150000.00,['Системный администратор']


In [9]:
df.head(2)

,id,salary_from,salary_to,area,experience,employment,schedule,work_format,professional_role,is_premium,has_test,accept_temporary,is_gross,has_usd_in_description,published_at,description,name,salary_mid
0,132031395,700000.0,1200000.0,Алматы,between1And3,full,fullDay,"['ON_SITE', 'HYBRID']",['Дата-сайентист'],False,False,False,False,False,2026-04-10 16:02:39+03:00,Мы разрабатываем и развиваем продукт в сфере у...,ML Engineer / CV Engineer,950000.0
2,130648164,400000.0,800000.0,Астана,between1And3,full,fullDay,['ON_SITE'],['Дата-сайентист'],False,False,False,False,False,2026-03-20 15:49:17+03:00,Обязанности: • Разработка и улучшение NLP/LLM ...,ML Engineer (NLP / RAG),600000.0


In [10]:
df['work_format'] = df['work_format'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])
df['professional_role'] = df['professional_role'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else [])

df['role'] = df['professional_role'].apply(lambda x: x[0] if x else None)
df[['professional_role', 'role', 'work_format']].head(3)

,professional_role,role,work_format
0,[Дата-сайентист],Дата-сайентист,"[ON_SITE, HYBRID]"
2,[Дата-сайентист],Дата-сайентист,[ON_SITE]
3,[Дата-сайентист],Дата-сайентист,[REMOTE]


In [11]:
df[['professional_role', 'role', 'work_format']]

,professional_role,role,work_format
0,[Дата-сайентист],Дата-сайентист,"[ON_SITE, HYBRID]"
2,[Дата-сайентист],Дата-сайентист,[ON_SITE]
3,[Дата-сайентист],Дата-сайентист,[REMOTE]
4,[Дата-сайентист],Дата-сайентист,[ON_SITE]
5,[Дата-сайентист],Дата-сайентист,[ON_SITE]
...,...,...,...
1519,[DevOps-инженер],DevOps-инженер,[ON_SITE]
1520,[Администратор],Администратор,[REMOTE]
1521,[Системный администратор],Системный администратор,[ON_SITE]
1522,[Системный администратор],Системный администратор,[ON_SITE]


In [12]:
df['experience'].unique()

<StringArray>
['between1And3', 'moreThan6', 'between3And6', 'noExperience']
Length: 4, dtype: str

In [13]:
exp = {
    'noExperience': 'Junior (without experience)',
    'between1And3': 'Middle (1-3 years)',
    'between3And6': 'Senior (3-6 years)',
    'moreThan6': 'Lead (6+ years)'
}

df['experience_label'] = df['experience'].map(exp)

print(df['experience_label'].value_counts())
print(df['experience_label'].isna().sum())

experience_label
Senior (3-6 years)             723
Middle (1-3 years)             599
Lead (6+ years)                122
Junior (without experience)     79
Name: count, dtype: int64
0


In [14]:
df['role'].unique()

<StringArray>
[                                                'Дата-сайентист',
                                                    'Инженер ПНР',
                                       'Программист, разработчик',
                                     'Технический директор (CTO)',
                                           'Продуктовый аналитик',
                                                 'DevOps-инженер',
                                   'BI-аналитик, аналитик данных',
                                                     'Архитектор',
                                                'Бизнес-аналитик',
                                                         'Другое',
                                            'Маркетолог-аналитик',
                                             'Системный аналитик',
                             'Менеджер по компенсациям и льготам',
                                  'Руководитель отдела аналитики',
                    'Менеджер по маркетингу, инт

In [15]:
# IT_ROLES is a determined by hh.kz list of professional_roles, here we just pick up only it related roles
IT_ROLES = [
    'Дата-сайентист',
    'Программист, разработчик',
    'Продуктовый аналитик',
    'DevOps-инженер',
    'BI-аналитик, аналитик данных',
    'Архитектор',
    'Бизнес-аналитик',
    'Другое',
    'Системный аналитик',
    'Системный администратор',
    'Аналитик',
    'Тестировщик',
    'Руководитель группы разработки',
    'Специалист по информационной безопасности',
    'Гейм-дизайнер',
    'Технический директор (CTO)',
    'Системный инженер',
]

# Same list as in first notebook, but with clarification - what kind of analysts we need
# and \b adds word boundaries to avoid false matches like "development" != "developer"
# engineer is specified to avoid false matches like "Pre-Sales Engineer"
IT_ROLE_KEYWORDS = [
    r"\bразработчик\b", r"\bdeveloper\b",
    r"\bsoftware engineer\b", r"\bdata engineer\b", r"\bsystems engineer\b",
    r"\barchitect\b", r"\bархитектор\b",
    r"\bios\b", r"\bandroid\b", r"\bfrontend\b", r"\bbackend\b", r"\bfullstack\b",
    r"\bdevops\b", r"\bmlops\b", r"\bqa\b", r"\bтестировщик\b", r"\bадминистратор\b",
    r"\bfullstack\b", r"\bsecurity engineer\b", r"\bdba\b",
    r"\bпрограммист\b", r"\bdata scientist\b", r"\bmachine learning\b",
    r"\bсистемный аналитик\b", r"\bпродуктовый аналитик\b", r"\bbi-аналитик\b",
    r"\bаналитик данных\b", r"\bdata analyst\b", r"\bбизнес-аналитик\b",
    r"\b1с\b", r"database\s+engineer", r"soc\b", r"information\s+security", r"automation\s+engineer", r"ux\s+engineer",
    r"software delivery", r"ict\b", r"devops", r"system integration",
]

df_cleaned = df[df['role'].isin(IT_ROLES)]
other = df_cleaned['role'] == 'Другое'
if_it_name = df_cleaned['name'].str.lower().str.contains('|'.join(IT_ROLE_KEYWORDS), na=False, regex=True)
df_cleaned = df_cleaned[~other | if_it_name].copy()

print(f'before deleting {len(df)} rows\nafter deleting:')
print(len(df_cleaned))

before deleting 1523 rows
after deleting:
1177


In [16]:
print(df_cleaned[df_cleaned['role'] == 'Другое'])

             id  salary_from  salary_to       area    experience employment  \
66    131898390          NaN        NaN     Астана  between3And6       full   
261   131992717          NaN        NaN     Алматы  between1And3       full   
355   131255736          NaN        NaN     Алматы  between3And6       full   
395   130572281          NaN   700000.0     Алматы  between3And6       full   
448   132022744          NaN        NaN     Астана  between3And6       full   
455   132022820          NaN        NaN     Астана  between3And6       full   
490   131918935          NaN        NaN     Алматы  between3And6       full   
522   131957279     230000.0        NaN  Караганда  between1And3       full   
530   131892702          NaN        NaN     Атырау  between3And6       full   
575   132010747          NaN        NaN     Алматы  between1And3       full   
633   132052137     500000.0  1000000.0     Алматы  between3And6    project   
680   131744265          NaN        NaN     Астана  

In [17]:
# non-it vacancy 
df_cleaned.drop(df_cleaned[df_cleaned['id'] == 131892702].index, inplace=True)

In [18]:
df_labeled = df_cleaned[df_cleaned['salary_mid'].notna()].copy()
df_unlabeled = df_cleaned[df_cleaned['salary_mid'].isna()].copy()

print(f"Labeled data have {len(df_labeled)} vacancies")
print(f"Unlabeled data have {len(df_unlabeled)} vacancies")

Labeled data have 367 vacancies
Unlabeled data have 809 vacancies


In [19]:
df_cleaned.to_csv(f'../data/processed/vacancies_clean_{date_str}.csv', index=False)